# 第8回：クラスを予測する—分類

**今日の問い：正解率だけで十分なのはどんなときか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 混同行列とprecision・recall・F1・PR-AUCを利用場面へ結びつける
- 確率の較正（calibration）を信頼度図と指標で評価する
- 不均衡データへclass_weightや閾値調整で対処し、効果を検証する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- precision：陽性予測のうち正しかった割合
- recall：実際の陽性を見つけた割合
- PR-AUC：適合率-再現率曲線の下側面積
- 較正：予測確率と実際の頻度が一致している度合い
- class_weight：少数クラスの誤りを重く扱う設定

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 「正解率」だけ見ると、なぜ危ないのか

第5回で「多数派と答えるだけで正解率が高くなる」ことを見ました。この回はその続きで、
**正解率（accuracy）に代わる読み方**を身につけます。鍵になるのが4つの結果です。

- **真陽性(TP)**：活性を活性と当てた／**真陰性(TN)**：非活性を非活性と当てた
- **偽陽性(FP)**：非活性を活性と誤った（無駄な追試）／**偽陰性(FN)**：活性を見逃した（機会損失）

この4つを表にしたのが**混同行列**です。まずロジスティック回帰を学習し、各試料の**活性確率**を出します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
model = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]


## TRY：閾値0.5で混同行列と3指標を読む

確率が0.5以上なら「活性」と判定し、結果を混同行列で見ます。同時に3つの指標を出します。

- **precision（適合率）**：活性と判定したうち、本当に活性だった割合（＝空振りの少なさ）。
- **recall（再現率）**：本当の活性のうち、見つけられた割合（＝見逃しの少なさ）。
- **F1**：precisionとrecallのバランス（両方が高いときだけ高くなる）。


In [ ]:
prediction = (probability >= 0.5).astype(int)
print("accuracy:", round(accuracy_score(y_valid, prediction), 3))
print("precision:", round(precision_score(y_valid, prediction), 3))
print("recall:", round(recall_score(y_valid, prediction), 3))
print("F1:", round(f1_score(y_valid, prediction), 3))
ConfusionMatrixDisplay.from_predictions(y_valid, prediction, display_labels=["非活性", "活性"], cmap="Blues")
plt.title("混同行列")


### 出力の読み方

- 混同行列は**左上=TN、右下=TP**が当たり、**右上=FP、左下=FN**が外れ。色が濃い（数が多い）マスに注目します。
- **accuracyは高いのにrecallが低い**、という組み合わせが起きがちです。これは「非活性はよく当てるが、肝心の活性を見逃している」状態。活性が少ないデータでは、accuracyが良く見えてもこの罠にはまります。
- 「何を重視するか」で読む指標が変わる、というのがこの回の核心です。


## TRY：判定の「閾値」を動かしてみる

0.5は絶対ではありません。閾値を下げると「活性」と判定する数が増え、**recallは上がるがprecisionは下がる**、
というトレードオフが起きます。0.3・0.5・0.7で指標がどう動くか並べます。


In [ ]:
rows = []
for threshold in [0.3, 0.5, 0.7]:
    pred = (probability >= threshold).astype(int)
    rows.append({"閾値": threshold, "precision": precision_score(y_valid, pred), "recall": recall_score(y_valid, pred), "F1": f1_score(y_valid, pred)})
pd.DataFrame(rows).round(3)


### 出力の読み方

- 閾値を下げる（0.3）と**recallが上がりprecisionが下がる**、上げる（0.7）と逆。表で必ずこの向きになるはずです。
- **見逃しを避けたい場面は閾値を下げ、空振りを避けたい場面は上げる**。閾値はモデルの外側で、目的に合わせて選ぶダイヤルです（第5回のコスト最適閾値につながります）。


## CORE深掘り：不均衡データではPR-AUCを見る

「閾値をいくつにするか」を決める前に、モデルの**確率の質そのもの**を1つの数字で測りたい。不均衡データ
（活性が少ない）では、ROC-AUCよりも**PR-AUC（適合率-再現率曲線の面積）**の方が実態を映します。


In [ ]:
from sklearn.metrics import average_precision_score
print("活性の割合:", round(df["active"].mean(), 3))
print("PR-AUC(平均適合率):", round(average_precision_score(y_valid, probability), 3))
print("常に多数派と予測したときのaccuracy:", round((y_valid == y_valid.mode()[0]).mean(), 3))


### 出力の読み方

- 「活性の割合」が小さいのに「多数派予測のaccuracy」が高い——**accuracyの水増し**を数字で確認できます。
- **PR-AUC**は「活性の割合」を基準線とし、それを大きく上回るほど、モデルが活性をうまく上位に並べていると読めます。閾値を決めずにモデルの良さを比べたいときの主指標です。


## 話し合い

「見逃し（FN）と空振り（FP）の、どちらがこのテーマでは高くつくか？」を5人で言葉にします。
探索段階なら見逃しを嫌ってrecall寄り、確証段階なら空振りを嫌ってprecision寄り。**正解は場面で変わります。**


## DEEP DIVE：確率を「信じてよいか」と、不均衡対策

確率をコストの計算（第5回）に使うなら、その確率が**較正**されている——「0.8と言ったら本当に約80%」で
ある必要があります。ここでは較正の測り方と直し方、そして少数クラスへの対処を扱います。


### 較正：予測確率と実際の頻度は一致しているか

**信頼度図**は、予測確率（横軸）に対して実際の活性率（縦軸）を描き、対角線に近いほど較正が良い、と
読みます。**Brierスコア**は較正のズレを1つの数字にしたもの（小さいほど良い）。`CalibratedClassifierCV`で
較正し直し、前後を比べます。


In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

base_clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
cal_clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5).fit(X_train, y_train)
plt.figure(figsize=(6, 5))
for name, clf in {"未較正": base_clf, "較正後": cal_clf}.items():
    p = clf.predict_proba(X_valid)[:, 1]
    print(f"{name}: Brier={brier_score_loss(y_valid, p):.3f}（小さいほど良い）")
    frac, mean_pred = calibration_curve(y_valid, p, n_bins=5)
    plt.plot(mean_pred, frac, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("予測確率"); plt.ylabel("実際の頻度"); plt.legend(); plt.title("信頼度図")
plt.tight_layout()


### 出力の読み方

- 折れ線が**対角線（点線）に近い**ほど較正が良好。対角線から膨らんでいれば、その確率帯で自信過剰／過小です。
- Brierが較正後に下がっていれば改善成功。ただし小さいデータでは較正が不安定なこともあるので、図と数字の両方で判断します。


### コスト行列で閾値を決める（較正済み確率で）

第5回と同じ考え方を、較正した確率に適用します。見逃し(FN)が空振り(FP)の8倍高いとして、期待コストが
最小の閾値を探します。**較正済みの確率**を使うことで、コスト計算の前提が整います。


In [ ]:
import numpy as np
proba_cal = cal_clf.predict_proba(X_valid)[:, 1]
cost_fn, cost_fp = 8, 1
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba_cal >= t).astype(int)
    fp = int(((pred == 1) & (y_valid == 0)).sum())
    fn = int(((pred == 0) & (y_valid == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
table = pd.DataFrame(rows)
print("コスト最小の閾値:", table.loc[table["期待コスト"].idxmin(), "閾値"])
table


### 出力の読み方

見逃しのコストが高いので、最適閾値は**0.5より低め**に出ます。「0.5で判定」がいかに恣意的かを、
ここでも数字で確認できます。コスト比を変えれば最適点も動きます。


### class_weight：少数クラスの誤りを重く扱う

閾値調整とは別に、**学習の時点で**少数クラス（活性）の誤りを重く扱う方法があります。`class_weight="balanced"`は
少数クラスを自動で重み付けします。recall（見逃しの少なさ）がどう変わるかを見ます。


In [ ]:
for label, weight in {"weightなし": None, "balanced": "balanced"}.items():
    clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000, class_weight=weight)).fit(X_train, y_train)
    pred = clf.predict(X_valid)
    print(f"{label:10s} F1={f1_score(y_valid, pred):.3f}  recall={recall_score(y_valid, pred):.3f}")


### 出力の読み方

`balanced`にすると**recallが上がりやすい**（活性を見つけにいく）反面、precisionやF1は下がることもあります。
「閾値で調整」と「重みで調整」は似た効果を持つ別の道具。どちらが目的に合うかを、指標を見て選びます。


## よくある誤り

- 常に閾値0.5を使う
- 偽陽性と偽陰性のコストを同じとみなす
- 未較正の確率をそのまま意思決定へ使う

## SELF-STUDY（任意・30〜60分）

- CalibratedClassifierCVで較正前後の信頼度図とBrierスコアを比べる
- コスト行列から期待コスト最小の閾値を求め、0.5と比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. accuracyが危険な例は何か
2. 較正が悪い確率を使うと何が起きるか
3. コストから閾値をどう決めるか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
